V0 du traitement de donnée

In [57]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings("ignore")

In [58]:

FILE = "Rapports_Billing Account for veolia.com, 2026-01-01 — 2026-06-23.csv"
# FILE = "Rapports_Billing Account for veolia.com, 2026-01-01 — 2026-06-30.csv"

df = pd.read_csv(FILE, decimal=",")

# Conversion en datetime

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df["Mois"] = df["Date"].dt.to_period("M").astype(str)
df["Jour"] = df["Date"].dt.to_period("D").astype(str)
# Garder uniquement la date (sans heure)

df["Date"] = df["Date"].dt.date
df = df[df["Coût catalogue (€)"]>1]

df[["Mois", "Date", "Coût catalogue (€)"]].head(10)

,Mois,Date,Coût catalogue (€)
0,2026-06,2026-06-23,6.21
1,2026-06,2026-06-23,2.98
2,2026-06,2026-06-23,2.30
13,2026-06,2026-06-22,11.93
14,2026-06,2026-06-22,4.49
15,2026-06,2026-06-22,2.91
16,2026-06,2026-06-22,1.49
27,2026-06,2026-06-21,11.92
28,2026-06,2026-06-21,2.91
29,2026-06,2026-06-21,1.48


In [59]:

fig = px.line(
    df,
    x="Jour",
    y="Sous-total (€)",
    color="Description du service",
    markers=True,
    title="Coûts GCP par service — Billing Veolia (2026)",
    labels={"Jour": "Jour", "Sous-total (€)": "Coût (€)", "Description du service": "Service"},
)

fig.update_layout(
    xaxis_title="Jour",
    yaxis_title="Coût (€)",
    legend_title="Service",
    hovermode="x unified",
)

fig.show()


## 1. Statistiques descriptives globales

In [83]:
# Coût total par Jour (toutes lignes agrégées)
monthly = (
    df.groupby("Mois")["Sous-total (€)"]
    .sum()
    .sort_index()
    .rename("Total")
)
df

,Date,Description du service,ID du service,Coût catalogue (€),Économies négociées (€),Programmes de remises (€),Autres remises (€),Sous-total non arrondi (€),Sous-total (€),Mois,Jour
0,2026-06-23,BigQuery,24E6-581D-38E5,6.21,-1.06,0.0,0.00,5.153586,5.15,2026-06,2026-06-23
1,2026-06-23,Cloud SQL,9662-B51E-5089,2.98,-0.45,0.0,0.00,2.534922,2.53,2026-06,2026-06-23
2,2026-06-23,Claude Sonnet 4.5,5029-7A29-0F52,2.30,0.00,0.0,0.00,2.298581,2.30,2026-06,2026-06-23
13,2026-06-22,Cloud SQL,9662-B51E-5089,11.93,-1.79,0.0,0.00,10.140824,10.14,2026-06,2026-06-22
14,2026-06-22,BigQuery,24E6-581D-38E5,4.49,-0.76,0.0,0.00,3.730005,3.73,2026-06,2026-06-22
...,...,...,...,...,...,...,...,...,...,...,...
2020,2026-01-06,Cloud Spanner,CC63-0873-48FD,2.87,-0.43,0.0,-2.44,0.000090,0.00,2026-01,2026-01-06
2029,2026-01-06,Invoice,A656-35D2-EF7F,2.55,0.00,0.0,-2.55,0.000000,0.00,2026-01,2026-01-06
2030,2026-01-05,Vertex AI,C7E2-9256-1C43,8.33,-1.25,0.0,-7.08,0.000922,0.00,2026-01,2026-01-05
2031,2026-01-05,Cloud Spanner,CC63-0873-48FD,1.99,-0.30,0.0,-1.69,0.000215,0.00,2026-01,2026-01-05


In [72]:
# Pivot service × Jour (pour analyses par service)
pivot = (
    df.pivot_table(index="Mois", columns="Description du service",
                   values="Sous-total (€)", aggfunc="sum")
    .fillna(0)
    .sort_index()
)

s = monthly.values.astype(float)
n = len(s)

mean_   = np.mean(s)
median_ = np.median(s)
std_    = np.std(s, ddof=1)
cv_     = std_ / mean_ * 100
skew_   = stats.skew(s)
kurt_   = stats.kurtosis(s, fisher=True)   # excès de kurtosis (normale = 0)
q1, q3  = np.percentile(s, [25, 75])
iqr_    = q3 - q1
rng_    = s.max() - s.min()
mad_    = np.median(np.abs(s - median_))   # Median Absolute Deviation

print("══════════════════════════════════════════")
print("   STATISTIQUES DESCRIPTIVES — Coût Total")
print("══════════════════════════════════════════")
print(f"  N Jour          : {n}")
print(f"  Somme totale    : {s.sum():>10.2f} €")
print(f"  Moyenne (μ)     : {mean_:>10.2f} €")
print(f"  Médiane         : {median_:>10.2f} €")
print(f"  Écart-type (σ)  : {std_:>10.2f} €")
print(f"  CV (σ/μ)        : {cv_:>9.1f} %")
print(f"  Min / Max       : {s.min():>7.2f} € / {s.max():.2f} €")
print(f"  Range           : {rng_:>10.2f} €")
print(f"  Q1 / Q3         : {q1:>7.2f} € / {q3:.2f} €")
print(f"  IQR             : {iqr_:>10.2f} €")
print(f"  MAD             : {mad_:>10.2f} €")
print(f"  Skewness        : {skew_:>10.4f}  {'(asymétrie droite)' if skew_>0 else '(asymétrie gauche)'}")
print(f"  Kurtosis (excès): {kurt_:>10.4f}  {'(leptokurtique)' if kurt_>0 else '(platykurtique)'}")
print("══════════════════════════════════════════")

# Boxplot + distribution
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Boxplot mensuel", "Distribution des coûts"])

fig.add_trace(go.Box(y=s, name="Coût total", boxpoints="all",
                     marker_color="steelblue", jitter=0.4, pointpos=-1.8), row=1, col=1)

fig.add_trace(go.Histogram(x=s, nbinsx=8, name="Distribution",
                           marker_color="steelblue", opacity=0.8), row=1, col=2)

for val, label, color in [(mean_, f"μ = {mean_:.0f}€", "red"),
                           (median_, f"Md = {median_:.0f}€", "orange")]:
    fig.add_vline(x=val, line_color=color, line_dash="dash",
                  annotation_text=label, row=1, col=2)

fig.update_layout(title="Distribution du coût mensuel total", showlegend=False, height=420)
fig.show()

══════════════════════════════════════════
   STATISTIQUES DESCRIPTIVES — Coût Total
══════════════════════════════════════════
  N Jour          : 6
  Somme totale    :    3327.50 €
  Moyenne (μ)     :     554.58 €
  Médiane         :     591.43 €
  Écart-type (σ)  :     178.76 €
  CV (σ/μ)        :      32.2 %
  Min / Max       :  258.98 € / 720.49 €
  Range           :     461.51 €
  Q1 / Q3         :  475.81 € / 693.79 €
  IQR             :     217.98 €
  MAD             :     119.46 €
  Skewness        :    -0.6670  (asymétrie gauche)
  Kurtosis (excès):    -0.8587  (platykurtique)
══════════════════════════════════════════


## 2. Tendance — Régression linéaire & moyenne mobile

In [86]:
df["Coût catalogue (€)"].array.astype(float)

array([ 6.21,  2.98,  2.3 , 11.93,  4.49,  2.91,  1.49, 11.92,  2.91,
        1.48, 11.92,  2.91,  1.48, 11.92,  8.41,  2.84,  2.91,  2.11,
        1.49, 11.93,  3.15,  2.91,  1.49,  1.34, 12.97, 11.92,  7.97,
        2.91,  1.61,  1.49, 11.93,  8.52,  6.1 ,  2.91,  1.87,  1.49,
       11.92, 11.91,  4.55,  2.91,  1.43,  1.49, 11.92,  4.87,  2.91,
        1.48,  1.34, 11.92,  2.91,  1.48, 11.93,  6.45,  2.91,  2.71,
        1.49, 15.29, 11.92,  8.47,  2.91,  1.48, 11.92,  3.46,  2.91,
        2.07,  1.49, 11.92,  4.78,  3.43,  2.91,  1.49, 11.92,  6.21,
        2.91,  2.29,  1.5 ,  1.19, 11.92,  2.91,  2.2 ,  1.49,  1.08,
       11.92,  2.91,  1.48, 11.92,  9.  ,  6.77,  2.91,  1.5 , 11.92,
        3.75,  2.91,  1.5 , 11.92,  8.1 ,  3.56,  2.91,  1.5 ,  1.13,
       12.78, 14.98, 11.92,  2.91,  2.44,  1.49, 11.92,  5.45,  2.7 ,
        2.91,  1.79,  1.5 , 11.85,  2.55,  2.89,  1.66,  1.48, 11.85,
        2.89,  1.48, 13.53, 11.85,  8.24,  2.89,  2.32,  1.5 , 12.6 ,
       11.85, 11.85,

In [87]:
t = np.arange(int(df.shape[0])) 
s_2 = df["Coût catalogue (€)"].array.astype(float)
slope, intercept, r, p_val, se = stats.linregress(t, s_2)
trend_line = slope * t + intercept

# Intervalles de confiance 95% sur la régression
t_crit = stats.t.ppf(0.975, df=int(df.shape[0]) - 2)
s_err  = np.sqrt(np.sum((s_2 - trend_line) ** 2) / (int(df.shape[0]) - 2))
ci     = t_crit * s_err * np.sqrt(1/int(df.shape[0]) + (t - t.mean())**2 / np.sum((t - t.mean())**2))

# Moyenne mobile 3 Jour (centrée)
mm3 = pd.Series(s_2).rolling(3, center=True).mean().values

dates = monthly.index

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=s_2, mode="lines+markers",
                         name="Coût réel", line=dict(color="steelblue", width=2)))
fig.add_trace(go.Scatter(x=dates, y=trend_line, mode="lines",
                         name=f"Tendance linéaire (β={slope:+.1f}€/Jour)",
                         line=dict(color="red", dash="dash", width=2)))
fig.add_trace(go.Scatter(
    x=np.concatenate([dates, dates[::-1]]),
    y=np.concatenate([trend_line + ci, (trend_line - ci)[::-1]]),
    fill="toself", fillcolor="rgba(255,0,0,0.1)",
    line=dict(color="rgba(255,255,255,0)"),
    name="IC 95% régression", showlegend=True))
fig.add_trace(go.Scatter(x=dates, y=mm3, mode="lines",
                         name="Moy. mobile 3 Jour",
                         line=dict(color="orange", dash="dot", width=2)))

fig.update_layout(title=f"Tendance — β={slope:+.2f}€/Jour | R²={r**2:.3f} | p={p_val:.4f}",
                  xaxis_title="Jour", yaxis_title="Coût total (€)",
                  hovermode="x unified", height=440)
fig.show()

print(f"\n  Pente (β)     : {slope:+.2f} €/Jour")
print(f"  R²            : {r**2:.4f}  ({'tendance significative' if p_val < 0.05 else 'tendance non significative'}, p={p_val:.4f})")
print(f"  Projection M+1: {trend_line[-1] + slope:.2f} €")


  Pente (β)     : +0.00 €/Jour
  R²            : 0.0037  (tendance non significative, p=0.1052)
  Projection M+1: 6.24 €


## 3. Stationnarité — Tests ADF & KPSS

In [62]:
# ── ADF (H0 : racine unitaire = non stationnaire) ────────────────────────────
adf_stat, adf_p, adf_lags, _, adf_cv, _ = adfuller(s, autolag="AIC")

# ── KPSS (H0 : stationnaire) ─────────────────────────────────────────────────
kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(s, regression="c", nlags="auto")

print("══════════════════════════════════════════════════")
print("   TEST ADF  (H₀ : non stationnaire / racine unit.)")
print("══════════════════════════════════════════════════")
print(f"  Statistique   : {adf_stat:.4f}")
print(f"  p-value       : {adf_p:.4f}  →  {'STATIONNAIRE ✓' if adf_p < 0.05 else 'NON STATIONNAIRE ✗'}")
for k, v in adf_cv.items():
    print(f"  Val. critique {k}: {v:.4f}")

print()
print("══════════════════════════════════════════════════")
print("   TEST KPSS (H₀ : stationnaire)")
print("══════════════════════════════════════════════════")
print(f"  Statistique   : {kpss_stat:.4f}")
print(f"  p-value       : {kpss_p:.4f}  →  {'NON STATIONNAIRE ✗' if kpss_p < 0.05 else 'STATIONNAIRE ✓'}")
for k, v in kpss_cv.items():
    print(f"  Val. critique {k}: {v:.4f}")

print()
adf_ok  = adf_p  < 0.05
kpss_ok = kpss_p >= 0.05
if adf_ok and kpss_ok:
    verdict = "Série STATIONNAIRE (ADF + KPSS concordent)"
elif not adf_ok and not kpss_ok:
    verdict = "Série NON STATIONNAIRE — différenciation recommandée"
else:
    verdict = "Résultats contradictoires — possible stationnarité de tendance (TS)"
print(f"  Verdict       : {verdict}")

# ── ACF / PACF (auto-corrélations) ───────────────────────────────────────────
nlags = min(n - 2, 5)
acf_vals  = acf(s,  nlags=nlags, fft=True)
pacf_vals = pacf(s, nlags=nlags)
lags      = list(range(nlags + 1))
ci_bound  = 1.96 / np.sqrt(n)

fig = make_subplots(rows=1, cols=2, subplot_titles=["ACF", "PACF"])
for vals, col, name in [(acf_vals, 1, "ACF"), (pacf_vals, 2, "PACF")]:
    fig.add_trace(go.Bar(x=lags, y=vals, name=name,
                         marker_color=["red" if abs(v) > ci_bound else "steelblue"
                                       for v in vals]), row=1, col=col)
    for sign in [1, -1]:
        fig.add_hline(y=sign * ci_bound, line_dash="dash",
                      line_color="orange", row=1, col=col)

fig.update_layout(title="Auto-corrélations (ACF) et auto-corrélations partielles (PACF)",
                  showlegend=False, height=380)
fig.show()

══════════════════════════════════════════════════
   TEST ADF  (H₀ : non stationnaire / racine unit.)
══════════════════════════════════════════════════
  Statistique   : -3.2719
  p-value       : 0.0162  →  STATIONNAIRE ✓
  Val. critique 1%: -3.4730
  Val. critique 5%: -2.8803
  Val. critique 10%: -2.5767

══════════════════════════════════════════════════
   TEST KPSS (H₀ : stationnaire)
══════════════════════════════════════════════════
  Statistique   : 0.9409
  p-value       : 0.0100  →  NON STATIONNAIRE ✗
  Val. critique 10%: 0.3470
  Val. critique 5%: 0.4630
  Val. critique 2.5%: 0.5740
  Val. critique 1%: 0.7390

  Verdict       : Résultats contradictoires — possible stationnarité de tendance (TS)


## 4. Décomposition STL (Tendance / Saisonnalité / Résidus)

In [63]:
# STL nécessite period ≤ n//2 ; avec 6 Jour on prend period=3 (cycle trimestriel)
stl    = STL(monthly, period=3, robust=True)
result = stl.fit()

stl_trend    = result.trend
stl_seasonal = result.seasonal
stl_resid    = result.resid

# Force saisonnière : part de la variance expliquée par le composant saisonnier
var_total    = np.var(s)
var_seasonal = np.var(stl_seasonal.values)
var_trend    = np.var(stl_trend.values)
var_resid    = np.var(stl_resid.values)
fs = max(0.0, 1 - var_resid / (var_seasonal + var_resid))
ft = max(0.0, 1 - var_resid / (var_trend    + var_resid))

print(f"  Force de saisonnalité (Fs) : {fs:.4f}  {'(forte)' if fs > 0.6 else '(modérée)' if fs > 0.3 else '(faible)'}")
print(f"  Force de tendance     (Ft) : {ft:.4f}  {'(forte)' if ft > 0.6 else '(modérée)' if ft > 0.3 else '(faible)'}")
print(f"  Variance résiduelle        : {var_resid:.2f} €²  ({var_resid/var_total*100:.1f}% du total)")

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=["Série originale", "Tendance STL",
                                    f"Saisonnalité (Fs={fs:.2f})", "Résidus"])

colors = ["steelblue", "red", "green", "grey"]
series = [s, stl_trend.values, stl_seasonal.values, stl_resid.values]
modes  = ["lines+markers", "lines", "lines+markers", "lines+markers"]

for i, (vals, color, mode) in enumerate(zip(series, colors, modes), start=1):
    fig.add_trace(go.Scatter(x=dates, y=vals, mode=mode,
                             line=dict(color=color, width=1.8),
                             marker=dict(size=5)), row=i, col=1)

# Ligne zéro sur les résidus
fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1, row=4, col=1)

fig.update_layout(title="Décomposition STL — Tendance / Saisonnalité / Résidus",
                  showlegend=False, height=700)
fig.show()

  Force de saisonnalité (Fs) : 0.1284  (faible)
  Force de tendance     (Ft) : 0.4743  (modérée)
  Variance résiduelle        : 52.12 €²  (54.3% du total)


## 5. Détection d'anomalies — Z-score & IQR

In [65]:
z_scores   = np.abs(stats.zscore(s))
iqr_lower  = q1 - 1.5 * iqr_
iqr_upper  = q3 + 1.5 * iqr_

anomalies_z   = monthly[z_scores > 2].index.tolist()
anomalies_iqr = monthly[(s < iqr_lower) | (s > iqr_upper)].index.tolist()
all_anomalies = sorted(set(anomalies_z) | set(anomalies_iqr))

print("  Anomalies Z-score > 2  :", [str(d) for d in anomalies_z]   or "aucune")
print("  Anomalies IQR fence    :", [str(d) for d in anomalies_iqr] or "aucune")

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=s, mode="lines+markers",
                         name="Coût réel", line=dict(color="steelblue")))

# Bandes ±2σ
fig.add_hrect(y0=mean_ - 2*std_, y1=mean_ + 2*std_,
              fillcolor="rgba(255,165,0,0.12)", line_width=0,
              annotation_text="±2σ", annotation_position="top left")
fig.add_hline(y=mean_, line_dash="dash", line_color="red",
              annotation_text=f"μ = {mean_:.0f}€")

# Points anomalies
if all_anomalies:
    anom_vals = [monthly[d] for d in all_anomalies]
    fig.add_trace(go.Scatter(x=all_anomalies, y=anom_vals,
                             mode="markers", name="Anomalie",
                             marker=dict(color="red", size=14, symbol="x")))

fig.update_layout(title="Détection d'anomalies — bandes ±2σ & IQR fence",
                  xaxis_title="Jour", yaxis_title="Coût (€)",
                  hovermode="x unified", height=420)
fig.show()

# Heatmap de variation mensuelle MoM par service
mom = pivot.pct_change() * 100
mom_clean = mom.dropna().replace([np.inf, -np.inf], np.nan).fillna(0)

fig2 = px.imshow(mom_clean.T,
                 color_continuous_scale="RdYlGn",
                 color_continuous_midpoint=0,
                 title="Variation MoM (%) par service — Heatmap",
                 labels=dict(x="Jour", y="Service", color="Δ%"),
                 aspect="auto", height=500)
fig2.show()

  Anomalies Z-score > 2  : ['2026-02-16', '2026-03-10', '2026-03-11', '2026-06-02']
  Anomalies IQR fence    : ['2026-02-16']


## 6. Pareto & Volatilité par service

In [66]:
svc_rows = []
for svc in pivot.columns:
    vals        = pivot[svc].values
    active_vals = vals[vals > 0]
    if len(active_vals) == 0:
        continue
    svc_rows.append({
        "Service":   svc,
        "Total (€)": float(np.sum(vals)),
        "Moy. (€)":  float(np.mean(active_vals)),
        "Méd. (€)":  float(np.median(active_vals)),
        "Std (€)":   float(np.std(active_vals, ddof=1)) if len(active_vals) > 1 else 0.0,
        "CV (%)":    float(np.std(active_vals, ddof=1) / np.mean(active_vals) * 100)
                     if len(active_vals) > 1 and np.mean(active_vals) > 0 else 0.0,
        "Mois actifs": int(np.sum(vals > 0)),
    })

svc_df = pd.DataFrame(svc_rows).sort_values("Total (€)", ascending=False).reset_index(drop=True)
svc_df["Part (%)"]   = svc_df["Total (€)"] / svc_df["Total (€)"].sum() * 100
svc_df["Cumulé (%)"] = svc_df["Part (%)"].cumsum()

print(f"{'Service':<28} {'Total':>8}  {'Moy':>7}  {'Méd':>7}  {'Std':>7}  {'CV':>6}  {'Part':>5}  {'Cumulé':>6}")
print("-" * 85)
for _, row in svc_df.iterrows():
    marker = " ← 80%" if (row["Cumulé (%)"] >= 80 and (row["Cumulé (%)"] - row["Part (%)"]) < 80) else ""
    print(f"{row['Service']:<28} {row['Total (€)']:>8.2f}  {row['Moy. (€)']:>7.2f}  "
          f"{row['Méd. (€)']:>7.2f}  {row['Std (€)']:>7.2f}  {row['CV (%)']:>5.1f}%  "
          f"{row['Part (%)']:>5.1f}%  {row['Cumulé (%)']:>6.1f}%{marker}")

top80 = svc_df.loc[svc_df["Cumulé (%)"].shift(1, fill_value=0) < 80, "Service"].tolist()
print(f"\n  Règle 80/20 → {len(top80)} service(s) représentent 80% du coût : {', '.join(top80)}")

# Pareto chart
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=svc_df["Service"], y=svc_df["Part (%)"],
                     marker_color="steelblue", opacity=0.85, name="Part (%)",
                     text=[f"{v:.1f}%" for v in svc_df["Part (%)"]],
                     textposition="outside"), secondary_y=False)
fig.add_trace(go.Scatter(x=svc_df["Service"], y=svc_df["Cumulé (%)"],
                         mode="lines+markers", name="Cumulé (%)",
                         line=dict(color="red", width=2)), secondary_y=True)
fig.add_trace(go.Scatter(x=svc_df["Service"], y=[80]*len(svc_df),
                         mode="lines", name="Seuil 80%",
                         line=dict(color="orange", dash="dash", width=1.5)), secondary_y=True)
fig.update_yaxes(title_text="Part individuelle (%)", secondary_y=False)
fig.update_yaxes(title_text="Cumulé (%)", range=[0, 115], secondary_y=True)
fig.update_layout(title="Diagramme de Pareto — Concentration des coûts par service",
                  xaxis_tickangle=-30, height=480)
fig.show()

# Volatilité CV
cv_df = svc_df[svc_df["CV (%)"] > 0].sort_values("CV (%)", ascending=False)
fig2  = px.bar(cv_df, x="Service", y="CV (%)",
               title="Volatilité par service — Coefficient de Variation (σ/μ × 100)",
               color="CV (%)", color_continuous_scale="Reds",
               text=[f"{v:.1f}%" for v in cv_df["CV (%)"]],
               height=420)
fig2.update_traces(textposition="outside")
fig2.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False)
fig2.show()

Service                         Total      Moy      Méd      Std      CV   Part  Cumulé
-------------------------------------------------------------------------------------
Cloud SQL                     1100.59    10.01    10.08     0.79    7.9%   33.1%    33.1%
BigQuery                       623.62     6.05     4.12     6.72  111.0%   18.7%    51.8%
Claude Sonnet 4.6              449.91     5.23     4.64     2.76   52.8%   13.5%    65.3%
Cloud Spanner                  386.62     2.45     2.46     0.05    2.1%   11.6%    77.0%
Cloud Run                      344.38     2.48     2.91     0.84   33.9%   10.3%    87.3% ← 80%
Vertex AI                      288.61    10.31    11.39     3.03   29.3%    8.7%    96.0%
Claude Sonnet 4.5              122.13     2.71     1.83     2.39   88.1%    3.7%    99.7%
Claude Opus 4.5                 11.64     2.91     1.80     2.78   95.7%    0.3%   100.0%

  Règle 80/20 → 5 service(s) représentent 80% du coût : Cloud SQL, BigQuery, Claude Sonnet 4.6, Clo